# RRN GPU training — persistent local Vimeo-90K LMDB cache
CPU verifies one real clip first. T4 stages the first two existing shards, begins the real Epoch 1, and overlaps later shard copies with training. Cached shards remain in `/content`; checkpoints resume the same formal run.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install lmdb pyspng


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 kB 21.8 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import json, os, subprocess
PROJECT = Path('/content/drive/MyDrive/train_ai/projects/RNN')
ADAPTER_REPO = 'https://github.com/koorock92/rrn-colab-adapter.git'
ADAPTER = Path('/content/rrn-colab-adapter')
UPSTREAM = Path('/content/RRN-upstream')
UPSTREAM_COMMIT = 'db440c5d363dfe6c11bdd6c82e63781b295e6211'
if not ADAPTER.exists():
    subprocess.run(['git', 'clone', '--depth', '1', ADAPTER_REPO, str(ADAPTER)], check=True)
else:
    subprocess.run(['git', '-C', str(ADAPTER), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(ADAPTER), 'reset', '--hard', 'origin/main'], check=True)
if not UPSTREAM.exists():
    subprocess.run(['git', 'clone', 'https://github.com/junpan19/RRN.git', str(UPSTREAM)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', UPSTREAM_COMMIT], check=True)
WORK = ADAPTER / 'src'
SHARDS = Path('/content/drive/MyDrive/train_ai/datasets/vimeo90k/vimeo90k_shards')
CACHE = Path('/content/rrn_shard_cache')
PREVIOUS_OUT = PROJECT / 'logs/sharded_gpu_full'
FULL_OUT = PROJECT / 'logs/sharded_gpu_crop256'
SMOKE_OUT = FULL_OUT
USE_PREVIOUS_BEST = False  # False = corrected-crop training starts from random weights.
LR_CROP_SIZE = 64       # Central LR 64x64; matching HR crop is 256x256 for x4.
TRAIN_BATCH_SIZE = 4    # Paper-equivalent total batch; benchmark before increasing.
TRAIN_EPOCHS = 70       # Cost-controlled correction fine-tune; extend after Vid4 improves.
NUM_WORKERS = max(2, min(6, (os.cpu_count() or 2) - 2))
PREFETCH_FACTOR = 4
print('Training:', {'lr_crop': LR_CROP_SIZE, 'hr_crop': LR_CROP_SIZE * 4, 'batch': TRAIN_BATCH_SIZE, 'epochs': TRAIN_EPOCHS})
print('DataLoader:', {'workers': NUM_WORKERS, 'prefetch_factor': PREFETCH_FACTOR, 'queued_batches': NUM_WORKERS * PREFETCH_FACTOR, 'cpu_count': os.cpu_count()})
manifest = json.loads((SHARDS / 'manifest.json').read_text())
print('train shards:', len(manifest['train_shards']))
print('train samples:', manifest['train_samples'])
print('validation samples:', manifest['validation_samples'])


Training: {'lr_crop': 64, 'hr_crop': 256, 'batch': 4, 'epochs': 70}
DataLoader: {'workers': 6, 'prefetch_factor': 4, 'queued_batches': 24, 'cpu_count': 12}
train shards: 8
train samples: 64612
validation samples: 100


In [ ]:
import sys, torch
print('Runtime:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    sys.path.insert(0, str(WORK))
    from vimeo_sharded_dataset import VimeoSeptupletDataset
    from upstream_rrn_adapter import build_rrn
    first = SHARDS / manifest['train_shards'][0]['name']
    dataset = VimeoSeptupletDataset(first, split='train', scale=4, crop_size=LR_CROP_SIZE * 4, max_samples=1, augment=False)
    low, target, name = dataset[0]
    if dataset.env is not None: dataset.env.close(); dataset.env = None
    model = build_rrn(UPSTREAM, 4, 128, 10).cpu().eval()
    with torch.inference_mode(): prediction = model(low.unsqueeze(0))
    assert prediction.shape == target.unsqueeze(0).shape and torch.isfinite(prediction).all()
    print('CPU_VERIFY_PASSED', name, tuple(prediction.shape))
else:
    print('GPU_READY', torch.cuda.get_device_name(0))
!df -h / /content/drive


## Batch throughput benchmark (safe; does not modify checkpoints)
Runs batches 1/2/4/8 with the corrected LR 64 (HR 256) crop. Batch 4 matches the paper's total batch; use clips/s and memory to decide whether batch 8 is worth testing.

In [4]:
RUN_BATCH_BENCHMARK = True  # Run after the corrected 300-step smoke test.
BENCHMARK_COMPILE = False
if RUN_BATCH_BENCHMARK:
    import subprocess, sys, torch
    cached = sorted(path for path in CACHE.glob('train-*.lmdb') if (path / 'data.mdb').is_file())
    if not cached:
        sys.path.insert(0, str(WORK))
        from train_rrn_sharded import ShardStager
        shard_meta = {item['name']: item for item in manifest['train_shards']}
        first_name = manifest['train_shards'][0]['name']
        stager = ShardStager(SHARDS, CACHE, shard_meta, prefetch_mib_s=0)
        try:
            stager.get(first_name)
        finally:
            stager.close()
        cached = [CACHE / first_name]
    checkpoint = PREVIOUS_OUT / 'best.pt'
    if USE_PREVIOUS_BEST and checkpoint.is_file():
        state = torch.load(checkpoint, map_location='cpu', weights_only=False)
        print('CHECKPOINT_STATE', {k: state.get(k) for k in ('epoch', 'shard_position', 'next_batch', 'global_step', 'best_psnr')}, flush=True)
    else:
        print('Benchmark uses random weights; throughput and memory results remain valid.', flush=True)
    cmd = [sys.executable, '-u', str(WORK / 'benchmark_rrn_batches.py'),
           '--upstream-dir', str(UPSTREAM), '--data', str(cached[0]),
           '--batches', '1,2,4,8', '--lr-crop-size', str(LR_CROP_SIZE),
           '--steps', '30', '--warmup', '5', '--workers', str(NUM_WORKERS),
           '--prefetch-factor', str(PREFETCH_FACTOR), '--decoder', 'auto']
    if USE_PREVIOUS_BEST and checkpoint.is_file():
        cmd += ['--checkpoint', str(checkpoint)]
    if BENCHMARK_COMPILE:
        cmd += ['--compile', '--compile-mode', 'default']
    print('Benchmark shard:', cached[0], flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError(f'Batch benchmark failed: {process.returncode}')

STAGE_START name=train-000.lmdb resumed_gib=0.00 total_gib=10.80
STAGING name=train-000.lmdb percent=0.2 copied_gib=0.02 total_gib=10.80 mib_s=11.9 eta_s=925
STAGING name=train-000.lmdb percent=1.2 copied_gib=0.13 total_gib=10.80 mib_s=30.5 eta_s=359
STAGING name=train-000.lmdb percent=3.5 copied_gib=0.38 total_gib=10.80 mib_s=53.6 eta_s=199
STAGING name=train-000.lmdb percent=4.7 copied_gib=0.51 total_gib=10.80 mib_s=54.4 eta_s=194
STAGING name=train-000.lmdb percent=5.9 copied_gib=0.63 total_gib=10.80 mib_s=54.5 eta_s=191
STAGING name=train-000.lmdb percent=7.0 copied_gib=0.76 total_gib=10.80 mib_s=53.5 eta_s=192
STAGING name=train-000.lmdb percent=8.8 copied_gib=0.95 total_gib=10.80 mib_s=58.4 eta_s=173
STAGING name=train-000.lmdb percent=10.1 copied_gib=1.09 total_gib=10.80 mib_s=59.6 eta_s=167
STAGING name=train-000.lmdb percent=11.6 copied_gib=1.26 total_gib=10.80 mib_s=60.6 eta_s=161
STAGING name=train-000.lmdb percent=12.8 copied_gib=1.38 total_gib=10.80 mib_s=59.8 eta_s=161
ST

In [ ]:
import subprocess as _subprocess
_original_popen = getattr(_subprocess.Popen, '_rrn_original', _subprocess.Popen)
def _rrn_fast_start_popen(args, *positional, **kwargs):
    if isinstance(args, list) and any('train_rrn_sharded.py' in str(item) for item in args):
        args = list(args)
        if '--compile' not in args:
            args += ['--compile', '--compile-mode', 'default']
        if '--startup-buffer-shards' in args:
            args[args.index('--startup-buffer-shards') + 1] = '1'
        print('FAST_START compile=default startup_buffer=1', flush=True)
    return _original_popen(args, *positional, **kwargs)
_rrn_fast_start_popen._rrn_original = _original_popen
_subprocess.Popen = _rrn_fast_start_popen
print('TRAIN_FAST_START_HOOK_READY')

## Formal-run smoke checkpoint (300 steps, resumable)
These are real Epoch 1 steps in the final output directory. The first two shards are staged before training; later copies overlap training. Continue the same checkpoint below.

In [7]:
import subprocess, sys, torch
last = SMOKE_OUT / 'last.pt'
start_step = 0
if last.exists():
    checkpoint = torch.load(last, map_location='cpu', weights_only=False)
    start_step = int(checkpoint.get('global_step', 0))
smoke_stop = start_step + 300
print(f'Resumable speed test: step {start_step:,} -> {smoke_stop:,}', flush=True)
cmd = [sys.executable, str(WORK / 'train_rrn_sharded.py'),
       '--upstream-dir', str(UPSTREAM), '--shards-root', str(SHARDS), '--cache-dir', str(CACHE), '--output', str(SMOKE_OUT),
       '--device', 'cuda', '--scale', '4', '--lr-crop-size', str(LR_CROP_SIZE), '--batch-size', str(TRAIN_BATCH_SIZE),
       '--workers', str(NUM_WORKERS), '--channels', '128', '--blocks', '10', '--epochs', str(TRAIN_EPOCHS),
       '--save-every', '100', '--max-steps', str(smoke_stop), '--decoder', 'auto', '--prefetch-factor', str(PREFETCH_FACTOR),
       '--startup-buffer-shards', '2', '--shard-prefetch-ahead', '2', '--shard-prefetch-mib-s', '0',
       '--profile-every', '100', '--keep-cache', '--validation-scope', 'epoch', '--gpu-downsample', '--amp']
if last.exists():
    cmd += ['--resume', str(last)]
elif USE_PREVIOUS_BEST:
    previous_best = PREVIOUS_OUT / 'best.pt'
    assert previous_best.is_file(), previous_best
    cmd += ['--init-weights', str(previous_best)]
else:
    print('Starting a fresh corrected-crop run with random weights.', flush=True)
print(' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)


Resumable speed test: step 0 -> 300
Starting a fresh corrected-crop run with random weights.
/usr/bin/python3 /content/rrn-colab-adapter/src/train_rrn_sharded.py --upstream-dir /content/RRN-upstream --shards-root /content/drive/MyDrive/train_ai/datasets/vimeo90k/vimeo90k_shards --cache-dir /content/rrn_shard_cache --output /content/drive/MyDrive/train_ai/projects/RNN/logs/sharded_gpu_crop256 --device cuda --scale 4 --lr-crop-size 64 --batch-size 4 --workers 6 --channels 128 --blocks 10 --epochs 70 --save-every 100 --max-steps 300 --decoder auto --prefetch-factor 4 --startup-buffer-shards 2 --shard-prefetch-ahead 2 --shard-prefetch-mib-s 0 --profile-every 100 --keep-cache --validation-scope epoch --gpu-downsample --amp


CompletedProcess(args=['/usr/bin/python3', '/content/rrn-colab-adapter/src/train_rrn_sharded.py', '--upstream-dir', '/content/RRN-upstream', '--shards-root', '/content/drive/MyDrive/train_ai/datasets/vimeo90k/vimeo90k_shards', '--cache-dir', '/content/rrn_shard_cache', '--output', '/content/drive/MyDrive/train_ai/projects/RNN/logs/sharded_gpu_crop256', '--device', 'cuda', '--scale', '4', '--lr-crop-size', '64', '--batch-size', '4', '--workers', '6', '--channels', '128', '--blocks', '10', '--epochs', '70', '--save-every', '100', '--max-steps', '300', '--decoder', 'auto', '--prefetch-factor', '4', '--startup-buffer-shards', '2', '--shard-prefetch-ahead', '2', '--shard-prefetch-mib-s', '0', '--profile-every', '100', '--keep-cache', '--validation-scope', 'epoch', '--gpu-downsample', '--amp'], returncode=0)

In [ ]:
import json
lines = (SMOKE_OUT / 'metrics.jsonl').read_text().splitlines()
recent = [json.loads(line) for line in lines[-20:]]
print('steps:', len(recent))
print('mean seconds/step:', sum(x['seconds'] for x in recent) / len(recent))
print('last:', SMOKE_OUT / 'last.pt', (SMOKE_OUT / 'last.pt').exists())
print('best:', SMOKE_OUT / 'best.pt', (SMOKE_OUT / 'best.pt').exists())
!df -h / /content/drive
!nvidia-smi --query-gpu=name,memory.used,utilization.gpu --format=csv


## Continue the corrected-crop fine-tune

In [4]:
CONFIRM_FULL_TRAINING = True
if not CONFIRM_FULL_TRAINING:
    print('Not started. First confirm the resumable 300-step formal smoke run.')
else:
    import subprocess, sys, math
    from collections import deque
    from html import escape
    from IPython.display import HTML, display
    last = FULL_OUT / 'last.pt'
    cmd = [sys.executable, str(WORK / 'train_rrn_sharded.py'),
           '--upstream-dir', str(UPSTREAM), '--shards-root', str(SHARDS), '--cache-dir', str(CACHE), '--output', str(FULL_OUT),
           '--device', 'cuda', '--scale', '4', '--lr-crop-size', str(LR_CROP_SIZE), '--batch-size', str(TRAIN_BATCH_SIZE),
           '--workers', str(NUM_WORKERS), '--channels', '128', '--blocks', '10', '--epochs', str(TRAIN_EPOCHS),
           '--save-every', '1000', '--metrics-flush-every', '100', '--log-every', '25',
           '--decoder', 'auto', '--prefetch-factor', str(PREFETCH_FACTOR),
           '--startup-buffer-shards', '2', '--shard-prefetch-ahead', '2', '--shard-prefetch-mib-s', '0',
           '--profile-every', '100', '--keep-cache', '--validation-scope', 'epoch', '--gpu-downsample', '--amp']
    if last.exists():
        cmd += ['--resume', str(last)]
    elif USE_PREVIOUS_BEST:
        previous_best = PREVIOUS_OUT / 'best.pt'
        assert previous_best.is_file(), previous_best
        cmd += ['--init-weights', str(previous_best)]
    else:
        print('Starting a fresh corrected-crop run with random weights.', flush=True)
    print(' '.join(cmd), flush=True)
    ui = {'epoch': '-', 'shard': '-', 'step': 0, 'epoch_batch': 0, 'epoch_batches': 1,
          'loss': '--', 'mae': '--', 'lr': '--', 'psnr': '--', 'best': '--',
          'data_s': '--', 'compute_s': '--', 'wall_s': '--', 'samples_s': '--',
          'eta': '--', 'checkpoint': 0, 'decoder': 'auto',
          'stage': 'Shard cache ready', 'stage_percent': 100.0, 'event': 'Starting trainer...'}
    recent_wall = deque(maxlen=20)
    status = display(HTML('Starting...'), display_id=True)
    def fields(line):
        return dict(token.split('=', 1) for token in line.split()[1:] if '=' in token)
    def finite_metric(value, digits=4):
        try:
            number = float(value)
            return f'{number:.{digits}f}' if math.isfinite(number) else '--'
        except (TypeError, ValueError):
            return '--'
    def duration(seconds):
        seconds = max(0, int(seconds))
        hours, seconds = divmod(seconds, 3600)
        minutes, seconds = divmod(seconds, 60)
        return f'{hours:02d}:{minutes:02d}:{seconds:02d}'
    def render(color='#22c55e'):
        percent = 100.0 * ui['epoch_batch'] / max(ui['epoch_batches'], 1)
        stage = '' if ui['stage_percent'] >= 100 else f"""
        <div style='margin-top:10px;color:#fbbf24'>{escape(ui['stage'])}</div>
        <div style='height:6px;background:#374151;border-radius:4px;margin-top:4px'>
        <div style='height:6px;width:{ui['stage_percent']:.2f}%;background:#f59e0b;border-radius:4px'></div></div>"""
        status.update(HTML(f"""<div style='font-family:ui-monospace,SFMono-Regular,Consolas,monospace;padding:16px;background:#111827;color:#e5e7eb;border-radius:10px;line-height:1.55'>
        <div style='font-size:15px;font-weight:700;color:#f9fafb'>RRN ×4 · Epoch {escape(str(ui['epoch']))} · Shard {escape(str(ui['shard']))} · Step {ui['step']:,}</div>
        <div>PSNR: <b style='color:#60a5fa'>{ui['psnr']}</b> · Best: <b style='color:#a78bfa'>{ui['best']}</b> · MAE: {ui['mae']} · Loss: {ui['loss']} · LR: {ui['lr']}</div>
        <div style='height:14px;background:#374151;border-radius:7px;margin:9px 0 5px'>
        <div style='height:14px;width:{percent:.2f}%;background:{color};border-radius:7px'></div></div>
        <div>{ui['epoch_batch']:,}/{ui['epoch_batches']:,} ({percent:.1f}%) · {ui['wall_s']} s/step · {ui['samples_s']} clips/s · ETA {ui['eta']}</div>
        <div style='color:#9ca3af'>Data {ui['data_s']}s · GPU compute {ui['compute_s']}s · decoder {escape(ui['decoder'])} · checkpoint step {ui['checkpoint']:,}</div>
        {stage}<div style='margin-top:8px;color:#9ca3af;font-size:12px'>{escape(ui['event'])}</div></div>"""))
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for raw in process.stdout:
        line = raw.strip()
        if not line:
            continue
        data = fields(line)
        if line.startswith('TRAIN '):
            ui.update(epoch=data['epoch'], shard=data['shard'], step=int(data['step']),
                      loss=data['loss'], mae=data.get('mae', '--'), lr=data.get('lr', '--'),
                      psnr=finite_metric(data.get('val_psnr')), best=finite_metric(data.get('best_psnr')),
                      data_s=data.get('data_s', '--'), compute_s=data.get('compute_s', '--'),
                      wall_s=data.get('wall_s', '--'), samples_s=data.get('samples_s', '--'),
                      checkpoint=int(data.get('checkpoint_step', ui['checkpoint'])))
            ui['epoch_batch'], ui['epoch_batches'] = map(int, data['epoch_batch'].split('/'))
            recent_wall.append(float(data['wall_s']))
            ui['eta'] = duration((ui['epoch_batches'] - ui['epoch_batch']) * sum(recent_wall) / len(recent_wall))
        elif line.startswith('STAGING '):
            ui['stage_percent'] = float(data['percent'])
            ui['stage'] = (f"Prefetch {data['name']}: {data['copied_gib']}/{data['total_gib']} GiB · "
                           f"{data['mib_s']} MiB/s · ETA {data['eta_s']} s")
        elif line.startswith(('STAGED ', 'CACHE_READY ')):
            ui['stage_percent'] = 100.0
            ui['stage'] = line
        elif line.startswith('CHECKPOINT '):
            ui['checkpoint'] = int(data['step'])
            ui['event'] = f"Checkpoint saved to Drive at step {ui['checkpoint']:,}"
        elif line.startswith('INPUT_PIPELINE '):
            ui['decoder'] = data.get('decoder', 'auto')
            ui['event'] = line
        elif line.startswith(('VALIDATION_COMPLETE ', 'EPOCH_COMPLETE ')):
            ui['psnr'] = finite_metric(data.get('psnr', data.get('validation_psnr')))
            ui['best'] = finite_metric(data.get('best_psnr'))
            ui['event'] = line
        elif line.startswith('SHARD_COMPLETE '):
            ui['psnr'] = finite_metric(data.get('validation_psnr'))
            ui['best'] = finite_metric(data.get('best_psnr'))
            ui['event'] = line
        elif line.startswith(('PREFETCH_', 'SHARD_', 'RESUMED', 'CACHE_', 'PROFILE')):
            ui['event'] = line
        else:
            print(line, flush=True)
        render()
    returncode = process.wait()
    if returncode:
        render('#d32f2f')
        raise subprocess.CalledProcessError(returncode, cmd)
    ui['event'] = 'Training finished successfully.'
    ui['train_percent'] = 100.0
    render('#2e7d32')


/usr/bin/python3 /content/rrn-colab-adapter/src/train_rrn_sharded.py --upstream-dir /content/RRN-upstream --shards-root /content/drive/MyDrive/train_ai/datasets/vimeo90k/vimeo90k_shards --cache-dir /content/rrn_shard_cache --output /content/drive/MyDrive/train_ai/projects/RNN/logs/sharded_gpu_crop256 --device cuda --scale 4 --lr-crop-size 64 --batch-size 4 --workers 6 --channels 128 --blocks 10 --epochs 70 --save-every 1000 --metrics-flush-every 100 --log-every 25 --decoder auto --prefetch-factor 4 --startup-buffer-shards 2 --shard-prefetch-ahead 2 --shard-prefetch-mib-s 0 --profile-every 100 --keep-cache --validation-scope epoch --gpu-downsample --amp --resume /content/drive/MyDrive/train_ai/projects/RNN/logs/sharded_gpu_crop256/last.pt


STAGE_START name=validation.lmdb resumed_gib=0.00 total_gib=2.70
STAGE_START name=train-002.lmdb resumed_gib=0.00 total_gib=10.80
STAGE_START name=train-004.lmdb resumed_gib=0.00 total_gib=10.80
STAGE_START name=train-001.lmdb resumed_gib=0.00 total_gib=10.80
STAGE_START name=train-006.lmdb resumed_gib=0.00 total_gib=10.80
STAGE_START name=train-007.lmdb resumed_gib=0.00 total_gib=10.80
STAGE_START name=train-005.lmdb resumed_gib=0.00 total_gib=10.80
STAGE_START name=train-000.lmdb resumed_gib=0.00 total_gib=10.80
STAGE_START name=train-003.lmdb resumed_gib=0.00 total_gib=10.80
TRAINING_STOPPED_AT_STEP=1130500 last=/content/drive/MyDrive/train_ai/projects/RNN/logs/sharded_gpu_crop256/last.pt best=/content/drive/MyDrive/train_ai/projects/RNN/logs/sharded_gpu_crop256/best.pt
